# SpendShield — Dataset Version Comparison

This notebook compares preserved v1 artifacts with revised v2 artifacts. It
does not select a model using test performance and makes no real fraud claim.


## Comparison criteria

The comparison covers version metadata, row/class/split distributions,
feature counts and validation means, shortcut rates, baseline metrics,
per-class recall, false-positive and false-negative behavior, reproducibility,
leakage, and temporal integrity.


In [1]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd()
for candidate in (ROOT, ROOT.parent, ROOT.parent.parent):
    if (candidate / "ml").is_dir() and (candidate / "data" / "synthetic").is_dir():
        ROOT = candidate
        break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

V2_DIR = ROOT / "data" / "synthetic" / "v2"
V2_FEATURE_DIR = V2_DIR / "features"
V2_BASELINE_DIR = V2_DIR / "baseline"
V2_ERROR_DIR = V2_DIR / "error_analysis"
V2_COMPARISON_DIR = ROOT / "data" / "synthetic" / "comparison"
from ml.dataset_comparison import compare_versions

comparison = compare_versions(
    ROOT / "data" / "synthetic",
    V2_DIR,
    V2_COMPARISON_DIR,
)
print(json.dumps({
    "decision": comparison["decision"],
    "old": {
        "version": comparison["old"]["dataset_version"],
        "rows": comparison["old"]["row_count"],
        "test_metrics": comparison["old"]["baseline_test_metrics"],
    },
    "new": {
        "version": comparison["new"]["dataset_version"],
        "rows": comparison["new"]["row_count"],
        "test_metrics": comparison["new"]["baseline_test_metrics"],
    },
    "metric_deltas": comparison["metric_deltas"],
    "false_positive_rate_against_normal": comparison["false_positive_rate_against_normal"],
    "reproducibility": comparison["reproducibility"],
    "leakage_checks": comparison["leakage_checks"],
}, indent=2))


{
  "decision": "REVISED_DATASET_READY_FOR_NEXT_ML_PHASE",
  "old": {
    "version": "v1",
    "rows": 10000,
    "test_metrics": {
      "accuracy": 0.862826,
      "classes": [
        "normal",
        "synthetic_behavior_deviation",
        "synthetic_combined_pattern",
        "synthetic_high_amount",
        "synthetic_rapid_repeat",
        "synthetic_unusual_time"
      ],
      "confusion_matrix": [
        [
          1079,
          28,
          4,
          21,
          0,
          0
        ],
        [
          26,
          8,
          0,
          2,
          0,
          0
        ],
        [
          5,
          0,
          7,
          6,
          0,
          1
        ],
        [
          26,
          1,
          5,
          82,
          0,
          0
        ],
        [
          0,
          0,
          0,
          0,
          52,
          0
        ],
        [
          74,
          1,
          0,
          0,
          0,
          30


In [2]:
for label in ("synthetic_behavior_deviation", "synthetic_combined_pattern", "synthetic_high_amount", "synthetic_rapid_repeat", "synthetic_unusual_time"):
    old_recall = comparison["old"]["baseline_test_metrics"]["per_class"][label]["recall"]
    new_recall = comparison["new"]["baseline_test_metrics"]["per_class"][label]["recall"]
    print(label, {"v1_recall": old_recall, "v2_recall": new_recall})


synthetic_behavior_deviation {'v1_recall': 0.222222, 'v2_recall': 0.0}
synthetic_combined_pattern {'v1_recall': 0.368421, 'v2_recall': 0.0}
synthetic_high_amount {'v1_recall': 0.719298, 'v2_recall': 0.189655}
synthetic_rapid_repeat {'v1_recall': 1.0, 'v2_recall': 1.0}
synthetic_unusual_time {'v1_recall': 0.285714, 'v2_recall': 0.0}


## Decision

The final decision is based on data quality and protocol evidence, not on
higher accuracy alone. The result remains a synthetic research artifact and
must not be used for production payment or fraud decisions.
